<a href="https://colab.research.google.com/github/your-org/alexpose/blob/main/experiments/multiple-sclerosis/05_representation_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 - Visualizing the learned representations

We have a trained encoder. What did it actually learn? In this notebook we turn each video into a single feature vector using the frozen target encoder, then project those vectors to two dimensions with t-SNE and UMAP so we can see whether normal, ms, and pd land in different regions.

We compare two moments: right after pretraining on normal only, and after progressive fine-tuning with VICReg. If the story holds, the clusters should be better separated after fine-tuning. On this small dataset the effect is a demonstration of mechanism, not proof, and we say so plainly.


In [ ]:
# --- Setup: install dependencies (Colab installs; local usually already has them) ---
import importlib, importlib.util, subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

def _need(mod):
    return importlib.util.find_spec(mod) is None

# Light deps used by every notebook.
_pkgs = []
for mod, pip_name in [('cv2','opencv-python'), ('mediapipe','mediapipe'),
                      ('sklearn','scikit-learn'), ('pandas','pandas'),
                      ('matplotlib','matplotlib'), ('tqdm','tqdm')]:
    if _need(mod):
        _pkgs.append(pip_name)
# torch is guarded so Colab's preinstalled GPU torch is never downgraded.
if _need('torch'):
    _pkgs.append('torch')
if _pkgs:
    print('installing:', _pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs])
else:
    print('all light dependencies already present')

In [ ]:
# --- Make `sjepa` and `ambient` importable, locally and in Colab ---
from pathlib import Path
import sys, subprocess

def _find_exp_dir():
    # Local run: this notebook sits in experiments/multiple-sclerosis.
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'sjepa' / '__init__.py').exists():
            return p
    return None

EXP_DIR = _find_exp_dir()
if EXP_DIR is None:
    # Colab: clone the repo, then point at the experiment folder.
    REPO = 'https://github.com/your-org/alexpose.git'  # <-- edit to your fork
    if not Path('alexpose').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO])
    EXP_DIR = Path('alexpose') / 'experiments' / 'multiple-sclerosis'

REPO_ROOT = EXP_DIR.parents[1]
for p in (str(EXP_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('experiment dir:', EXP_DIR)
print('repo root     :', REPO_ROOT)

In [ ]:
# --- Paths and profile (reads the root .env if python-dotenv is present) ---
import os
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass

VIDEO_DIR = EXP_DIR / 'video-data'
ARTIFACT_DIR = EXP_DIR / 'artifacts'
KEYPOINTS_DIR = ARTIFACT_DIR / 'keypoints'
IMAGES_DIR = EXP_DIR / 'images'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Pick the model size profile. 'laptop' is the fast default; set SJEPA_PROFILE=gpu
# in your .env for a larger model, or SJEPA_SMOKE=1 for a near-instant test run.
os.environ.setdefault('SJEPA_PROFILE', 'laptop')
print('SJEPA_PROFILE =', os.environ['SJEPA_PROFILE'],
      '| SJEPA_SMOKE =', os.environ.get('SJEPA_SMOKE', '0'))

In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(IMAGES_DIR / 'vicreg_clusters.svg')))

## Embed every video with the frozen encoder

For each video we average the encoder's features over its windows and over the masked joints, giving one vector per video. We do this for both checkpoints.


In [ ]:
import numpy as np, torch
from sjepa.config import get_config
from sjepa.models import build_model, pick_device
from sjepa.train import load_checkpoint
from sjepa.masking import AnatomicalMaskSampler
from sjepa.data import load_index, sliding_windows

cfg = get_config(); device = pick_device()
records = load_index(KEYPOINTS_DIR)
sampler = AnatomicalMaskSampler(cfg.num_joints, cfg.num_time_tokens)
tm = torch.from_numpy(sampler.target_mask).to(device)

def embed_all(ckpt):
    m = build_model(cfg, device=device)
    load_checkpoint(ckpt, m, map_location=device)
    vecs, labels = [], []
    for r in records:
        w = sliding_windows(r.load_norm(), cfg.window_frames, cfg.window_stride)
        x = torch.from_numpy(w).float().to(device)
        with torch.no_grad():
            vecs.append(m.embed(x, tm).mean(0).cpu().numpy())
        labels.append(r.label)
    return np.stack(vecs), labels

E_pre, y = embed_all(ARTIFACT_DIR / 'sjepa_pretrain_normal.pt')
E_ft, _ = embed_all(ARTIFACT_DIR / 'sjepa_finetuned_3class.pt')
np.savez(ARTIFACT_DIR / 'embeddings_3class.npz', E_pretrain=E_pre, E_finetune=E_ft,
         labels=np.array(y))
print('embedded', len(y), 'videos into', E_ft.shape[1], 'dimensions')

## Project and plot

t-SNE and UMAP both squeeze the high-dimensional vectors into a plane while trying to keep neighbors together. We color one hue per condition.


In [ ]:
from sklearn.manifold import TSNE
from sjepa.viz import scatter_2d
import matplotlib.pyplot as plt

def tsne2d(E):
    perp = min(15, max(2, len(E)//3))
    return TSNE(n_components=2, perplexity=perp, random_state=42, init='pca').fit_transform(E)

fig, ax = plt.subplots(1, 2, figsize=(11,4.4))
scatter_2d(tsne2d(E_pre), y, ax[0], 't-SNE: pretrain on normal only')
scatter_2d(tsne2d(E_ft), y, ax[1], 't-SNE: after fine-tune + VICReg')
plt.tight_layout(); plt.savefig(IMAGES_DIR / 'tsne_pretrain_vs_finetune.png', dpi=130)
plt.show()

In [ ]:
# UMAP view (falls back gracefully if umap-learn is missing).
import matplotlib.pyplot as plt
from sjepa.viz import scatter_2d
try:
    import umap
    def umap2d(E):
        nn = min(15, max(2, len(E)//3))
        return umap.UMAP(n_neighbors=nn, min_dist=0.3, random_state=42).fit_transform(E)
    fig, ax = plt.subplots(1, 2, figsize=(11,4.4))
    scatter_2d(umap2d(E_pre), y, ax[0], 'UMAP: pretrain only')
    scatter_2d(umap2d(E_ft), y, ax[1], 'UMAP: fine-tune + VICReg')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('UMAP not available, skipping:', e)

## Put a number on the separation

The silhouette score summarizes how tight and well separated the clusters are, from -1 (bad) to +1 (clean). We compare the two checkpoints. On 47 videos this number is noisy, so treat it as a hint, not a verdict.


In [ ]:
from sjepa.eval import silhouette
s_pre = silhouette(E_pre, y)
s_ft = silhouette(E_ft, y)
print(f'silhouette  pretrain-only: {s_pre:.3f}   fine-tuned+VICReg: {s_ft:.3f}')
if s_ft >= s_pre:
    print('Fine-tuning helped separate the clusters (as hoped).')
else:
    print('No clear gain here. On this tiny dataset that can happen; see the caveats in 06.')